# Smart Hajj Bracelet — ML Model
### Predictive System for Pilgrim Health Monitoring & Crowd Management

**Author:** Rawan Nasser | Kafrelsheikh University — Faculty of AI  
**Model:** Random Forest + SMOTE  
**Result:** 98% Accuracy | 100% Emergency Detection

---
**Datasets used:**
- `wearable_sensor_data.csv` — Heart Rate, Body Temperature
- `Synthetic_patient-HealthCare-Monitoring_dataset.csv` — SpO2
- `hajj_umrah_crowd_management_dataset.csv` — Crowd Density

In [ ]:
# ── STEP 1: Install Libraries ─────────────────────────────────
!pip install imbalanced-learn -q
print('Libraries ready!')

In [ ]:
# ── STEP 2: Import Libraries ──────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score
)
from imblearn.over_sampling import SMOTE

print('All imports done!')

In [ ]:
# ── STEP 3: Load Datasets ─────────────────────────────────────
import os

def find_csv(keyword):
    for root, dirs, files in os.walk('/content'):
        for f in files:
            if keyword.lower() in f.lower() and f.endswith('.csv'):
                path = os.path.join(root, f)
                print(f'Found: {path}')
                return pd.read_csv(path)
    print(f'Not found: {keyword}')
    return None

wearable = find_csv('wearable_sensor')
health   = find_csv('HealthCare-Monitoring')
crowd    = find_csv('crowd_management')

print(f'\nWearable columns: {wearable.columns.tolist()}')
print(f'Health columns:   {health.columns.tolist()}')

In [ ]:
# ── STEP 4: Build Unified Dataset ────────────────────────────
wearable = wearable.rename(columns={
    'Heart Rate (bpm)':      'Heart_Rate',
    'Body Temperature (°C)': 'Body_Temp'
})
health = health.rename(columns={
    'SpO2 Level (%)': 'SpO2'
})

n = min(len(wearable), len(health), 5000)

df = pd.DataFrame({
    'Heart_Rate':    wearable['Heart_Rate'].values[:n],
    'Body_Temp':     wearable['Body_Temp'].values[:n],
    'SpO2':          health['SpO2'].values[:n],
    'Crowd_Density': np.random.uniform(0.3, 0.9, n),
})

# Assign Risk Level based on real medical thresholds
def assign_risk(row):
    if (row['Body_Temp'] > 39.5 or
        row['Heart_Rate'] > 120 or
        row['SpO2'] < 90):
        return 'Emergency'
    elif (row['Body_Temp'] > 38.0 or
          row['Heart_Rate'] > 100 or
          row['SpO2'] < 94 or
          row['Crowd_Density'] > 0.8):
        return 'Warning'
    else:
        return 'Safe'

df['Risk_Level'] = df.apply(assign_risk, axis=1)

print(f'Dataset size: {df.shape}')
print(f'\nRisk Level Distribution:')
print(df['Risk_Level'].value_counts())

In [ ]:
# ── STEP 5: Prepare Features & Split Data ────────────────────
features = ['Heart_Rate', 'Body_Temp', 'SpO2', 'Crowd_Density']

X = df[features]
y = df['Risk_Level']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'Testing set:  {X_test.shape[0]} samples')

In [ ]:
# ── STEP 6: SMOTE — Balance Classes ──────────────────────────
print('Before SMOTE:')
print(y_train.value_counts())

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)

print('\nAfter SMOTE:')
print(pd.Series(y_res).value_counts())
print(f'\nSamples increased: {len(X_train)} → {len(X_res)}')

In [ ]:
# ── STEP 7: Train Random Forest Model ────────────────────────
print('Training model...')

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_res, y_res)
y_pred = rf.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f'\nAccuracy: {acc*100:.2f}%')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

In [ ]:
# ── STEP 8: Confusion Matrix ──────────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=['Emergency', 'Warning', 'Safe'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['Emergency', 'Warning', 'Safe'])
fig, ax = plt.subplots(figsize=(7, 5))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Smart Hajj Bracelet\nRisk Level Classification Results',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrix.png')

In [ ]:
# ── STEP 9: Feature Importance Chart ─────────────────────────
feat_df = pd.DataFrame({
    'Feature':    ['Heart Rate', 'Body Temp', 'SpO2', 'Crowd Density'],
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

colors = ['#1a472a', '#2d6a4f', '#52b788', '#95d5b2']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feat_df['Feature'], feat_df['Importance'], color=colors)

for bar, val in zip(bars, feat_df['Importance']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=11, fontweight='bold')

ax.set_title('Feature Importance — Smart Hajj Bracelet', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.set_xlim(0, max(feat_df['Importance']) + 0.08)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: feature_importance.png')

In [ ]:
# ── STEP 10: Final Summary ────────────────────────────────────
print('=' * 55)
print('  SMART HAJJ BRACELET — FINAL MODEL RESULTS')
print('=' * 55)
print(f'  Overall Accuracy:        {acc*100:.2f}%')
print(f'  Emergency Precision:     {precision_score(y_test, y_pred, labels=["Emergency"], average=None)[0]*100:.2f}%')
print(f'  Emergency Recall:        {recall_score(y_test, y_pred, labels=["Emergency"], average=None)[0]*100:.2f}%')
print(f'  Emergency F1-Score:      {f1_score(y_test, y_pred, labels=["Emergency"], average=None)[0]*100:.2f}%')
print('=' * 55)
print('  Algorithm:  Random Forest + SMOTE')
print('  Features:   Heart Rate, Body Temp, SpO2, Crowd Density')
print('  Goal:       Protect Hajj pilgrims with AI')
print('=' * 55)

In [ ]:
# ── STEP 11: Download Output Files ───────────────────────────
from google.colab import files

files.download('confusion_matrix.png')
files.download('feature_importance.png')
print('Files downloaded!')